In [13]:
import ccxt
import pandas as pd, numpy as np
import ta
from dotenv import load_dotenv
import os
from datetime import datetime, timezone

load_dotenv()

api_keys = {
    "apiKey": os.getenv("BINANCE_API_KEY"),
    "secret": os.getenv("BINANCE_SECRET_KEY"),
    "enableRateLimit": True,
}

binc = ccxt.binance(api_keys)

In [61]:
trades = binc.fetch_trades("ETH/USDC")
json_info = pd.DataFrame(trades)["info"]
trades_df = pd.json_normalize(json_info).drop(columns="M")
trades_df.columns = ["id", "price", "quantity", "f_id", "l_id", "timestamp", "sell"]


def convert_ts(ts):
    ts = ts / 1000
    return datetime.fromtimestamp(ts, tz=timezone.utc)


trades_df["timestamp"] = trades_df["timestamp"].apply(convert_ts)
trades_df["trades_n"] = trades_df["l_id"] - trades_df["f_id"] + 1
trades_df.set_index("timestamp", inplace=True)
trades_df.sort_index(ascending=False, inplace=True)
trades_df["price"] = trades_df["price"].astype("float")
trades_df["quantity"] = trades_df["quantity"].astype("float")
interval = "10s"
ohlcv = trades_df["price"].resample(interval).ohlc()

In [62]:
ohlcv.ffill(inplace=True)
ohlcv["SMA3"] = ohlcv["close"].rolling(window=3).mean()
ohlcv["SMA5"] = ohlcv["close"].rolling(window=5).mean()
ohlcv.dropna(inplace=True)

In [63]:
ohlcv

,open,high,low,close,SMA3,SMA5
timestamp,,,,,,
2026-05-10 10:26:20+00:00,2326.42,2326.42,2326.42,2326.42,2326.286667,2326.248
2026-05-10 10:26:30+00:00,2326.42,2326.42,2326.42,2326.42,2326.363333,2326.294
2026-05-10 10:26:40+00:00,2326.42,2326.42,2326.42,2326.42,2326.420000,2326.340
2026-05-10 10:26:50+00:00,2326.42,2326.87,2326.42,2326.87,2326.570000,2326.476
2026-05-10 10:27:00+00:00,2326.87,2327.01,2326.87,2326.89,2326.726667,2326.604
...,...,...,...,...,...,...
2026-05-10 10:40:30+00:00,2325.24,2325.24,2324.84,2324.84,2325.190000,2325.306
2026-05-10 10:40:40+00:00,2325.24,2325.24,2324.84,2324.84,2324.976667,2325.178
2026-05-10 10:40:50+00:00,2324.83,2324.83,2324.80,2324.83,2324.836667,2325.048


In [64]:
threshold = 0.0001

In [65]:
ohlcv['bull_crossover'] = ohlcv['SMA3'] > ohlcv['SMA5'] * (1 + threshold)
ohlcv['bear_crossover'] = ohlcv['SMA3'] < ohlcv['SMA5'] * (1 - threshold)

In [67]:
ohlcv[ohlcv['bear_crossover']]

,open,high,low,close,SMA3,SMA5,bull_crossover,bear_crossover
timestamp,,,,,,,,
2026-05-10 10:35:00+00:00,2326.19,2326.19,2326.19,2326.19,2326.306667,2326.584,False,True
2026-05-10 10:38:20+00:00,2325.49,2325.49,2325.49,2325.49,2325.583333,2325.910,False,True
